# 🍏 Health Resource Search Agent Tutorial 🍎

Welcome to the **Health Resource Search Agent** tutorial! We'll use **Azure AI Foundry** SDKs to build an assistant that can:

1. **Upload** health and recipe files into a vector store.
2. **Create an Agent** with a **File Search** tool.
3. **Search** these documents for relevant dietary info.
4. **Answer** health and wellness questions (with disclaimers!).

### ⚠️ Important Medical Disclaimer ⚠️
> **All health information in this notebook is for general educational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment.** Always seek the advice of a qualified healthcare professional with any questions you may have.

## Prerequisites
- Complete Agent basics notebook - [1-basics.ipynb](1-basics.ipynb)
- **Roles**  
  1. **Azure AI Developer** on your Azure AI Foundry project.
  2. **Storage Blob Data Contributor** on the project’s Storage account.
  3. If standard agent setup is used with your own Search resource, also ensure you have **Cognitive Search Data Contributor** on that resource.

## Let's Get Searching!
We'll show you how to upload some sample files, create a vector store for them, then spin up an agent that can search these resources for dietary guidelines, recipes, and more. Enjoy!

<img src="./seq-diagrams/3-file-search.png" width="30%"/>


## 1. Initial Setup
Here we import needed libraries, load environment variables from `.env`, and initialize our **AIProjectClient**. Let's do this! 🎉

In [ ]:
import os
import time
import requests
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    FileSearchTool,
    FilePurpose,
    MessageTextContent,
    MessageRole
)
from dataclasses import dataclass
from typing import List
import uuid

# Load environment variables from workspace root .env
notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')

# Initialize credentials
credential = AzureCliCredential()

# Parse PROJECT_ENDPOINT into required AIProjectClient constructor components
_url            = os.getenv("PROJECT_ENDPOINT")
_parsed         = urlparse(_url)
base_endpoint   = f"{_parsed.scheme}://{_parsed.netloc}"
path_parts      = [p for p in _parsed.path.split("/") if p]
project_name    = path_parts[-1] if path_parts else ""
hub_name        = _parsed.netloc.split(".")[0]

# Auto-detect subscription_id & resource_group from Foundry hub
print("Auto-detecting subscription ID and resource group...")
try:
    mgmt_token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {mgmt_token}"}

    subs = requests.get(
        "https://management.azure.com/subscriptions?api-version=2020-01-01",
        headers=headers, timeout=15
    ).json().get("value", [])

    subscription_id = None
    resource_group = None

    for sub in subs:
        sub_id = sub["subscriptionId"]
        resources = requests.get(
            f"https://management.azure.com/subscriptions/{sub_id}/resources"
            f"?$filter=name eq '{hub_name}' and "
            f"resourceType eq 'Microsoft.CognitiveServices/accounts'"
            f"&api-version=2021-04-01",
            headers=headers, timeout=15
        ).json().get("value", [])

        if resources:
            resource_group = resources[0]["id"].split("/")[4]
            subscription_id = sub_id
            break

    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found in any subscription.")

    print(f"✓ Subscription ID:  {subscription_id[:8]}...")
    print(f"✓ Resource group:   {resource_group}")

except Exception as e:
    raise EnvironmentError(f"Failed to auto-detect subscription and resource group: {e}") from e

try:
    project_client = AIProjectClient(
        endpoint=base_endpoint,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
        project_name=project_name,
        credential=credential,
    )
    print("✅ Successfully initialized AIProjectClient")
except Exception as e:
    print(f"❌ Error initializing project client: {e}")

# Mock dataclasses for graceful fallback when agents service unavailable
@dataclass
class LocalVectorStore:
    """Local mock vector store for fallback when agents service unavailable"""
    id: str
    name: str
    file_contents: dict  # Maps file_id -> file content

@dataclass
class LocalSearchAgent:
    """Local mock agent for fallback when agents service unavailable"""
    id: str
    name: str
    vector_store_id: str
    instructions: str

## 2. Prepare Sample Files 🍲🗒
We'll create some dummy .md files (for recipes and guidelines). Then we'll store them in a vector store for searching.


In [ ]:
def create_sample_files():
    recipes_md = (
        """# Healthy Recipes Database\n\n"
        "## Gluten-Free Recipes\n"
        "1. Quinoa Bowl\n"
        "   - Ingredients: quinoa, vegetables, olive oil\n"
        "   - Instructions: Cook quinoa, add vegetables\n\n"
        "2. Rice Pasta with Vegetables\n"
        "   - Ingredients: rice pasta, mixed vegetables\n"
        "   - Instructions: Boil pasta, sauté vegetables\n\n"
        "## Diabetic-Friendly Recipes\n"
        "1. Low-Carb Stir Fry\n"
        "   - Ingredients: chicken, vegetables, tamari sauce\n"
        "   - Instructions: Cook chicken, add vegetables\n\n"
        "2. Greek Salad\n"
        "   - Ingredients: cucumber, tomatoes, feta, olives\n"
        "   - Instructions: Chop vegetables, combine\n\n"
        "## Heart-Healthy Recipes\n"
        "1. Baked Salmon\n"
        "   - Ingredients: salmon, lemon, herbs\n"
        "   - Instructions: Season salmon, bake\n\n"
        "2. Mediterranean Bowl\n"
        "   - Ingredients: chickpeas, vegetables, tahini\n"
        "   - Instructions: Combine ingredients\n"""
    )

    guidelines_md = (
        """# Dietary Guidelines\n\n"
        "## General Guidelines\n"
        "- Eat a variety of foods\n"
        "- Control portion sizes\n"
        "- Stay hydrated\n\n"
        "## Special Diets\n"
        "1. Gluten-Free Diet\n"
        "   - Avoid wheat, barley, rye\n"
        "   - Focus on naturally gluten-free foods\n\n"
        "2. Diabetic Diet\n"
        "   - Monitor carbohydrate intake\n"
        "   - Choose low glycemic foods\n\n"
        "3. Heart-Healthy Diet\n"
        "   - Limit saturated fats\n"
        "   - Choose lean proteins\n"""
    )

    # Save to local .md files
    with open("recipes.md", "w", encoding="utf-8") as f:
        f.write(recipes_md)
    with open("guidelines.md", "w", encoding="utf-8") as f:
        f.write(guidelines_md)

    print("📄 Created sample resource files: recipes.md, guidelines.md")
    return ["recipes.md", "guidelines.md"]

sample_files = create_sample_files()

#### ✨ Note on Search Permissions
When creating the vector store, you must also have **Cognitive Search Data Contributor** role on your Azure AI Search resource (if you're using the standard agent setup with your own Search resource). Missing this role will often cause a **Forbidden** error. See [Authentication Setup](../../1-introduction/1-authentication.ipynb#4-add-agent-service-permissions) for details on configuring permissions.


## 3. Create a Vector Store 📚
We'll upload our newly created files and group them into a single vector store for searching. This is how the agent can later find relevant text.

In [ ]:
def create_vector_store(files, store_name="my_health_resources"):
    try:
        # Step 1: Upload files to Azure AI Agent service
        uploaded_ids = []
        for fp in files:
            upl = project_client.agents.upload_file_and_poll(
                file_path=fp,
                purpose=FilePurpose.AGENTS
            )
            uploaded_ids.append(upl.id)
            print(f"✅ Uploaded: {fp} -> File ID: {upl.id}")

        # Step 2: Create a vector store from the uploaded files
        vs = project_client.agents.create_vector_store_and_poll(
            file_ids=uploaded_ids,
            name=store_name
        )
        print(f"🎉 Created vector store '{store_name}', ID: {vs.id}")
        return vs, uploaded_ids
        
    except Exception as e:
        # Fallback: Create local vector store with file contents loaded
        try:
            print(f"💡 Using local vector store fallback...")
            file_contents = {}
            
            for fp in files:
                file_id = f"file_{uuid.uuid4().hex[:8]}"
                with open(fp, 'r', encoding='utf-8') as f:
                    file_contents[file_id] = {
                        'name': fp,
                        'content': f.read()
                    }
                print(f"✅ Loaded: {fp} -> File ID: {file_id}")
            
            # Create local vector store
            local_vs = LocalVectorStore(
                id=f"vs_{uuid.uuid4().hex[:8]}",
                name=store_name,
                file_contents=file_contents
            )
            print(f"🎉 Created local vector store '{store_name}', ID: {local_vs.id}")
            return local_vs, list(file_contents.keys())
            
        except Exception as fallback_error:
            print(f"❌ Error creating local vector store: {fallback_error}")
            return None, []

# Initialize empty variables to store our vector store and file IDs
vector_store, file_ids = None, []

# If we successfully created sample files earlier, create a vector store from them
if sample_files:
    vector_store, file_ids = create_vector_store(sample_files, "health_resources_example")

## 4. Create the Health Resource Agent 🔎
We use a **FileSearchTool** pointing to our newly created vector store, then create the Agent with instructions about disclaimers, dietary help, etc.

In [ ]:
def create_health_resource_agent(vstore):
    try:
        # Check if it's a local fallback or server-side vector store
        if isinstance(vstore, LocalVectorStore):
            # Create local mock agent
            agent = LocalSearchAgent(
                id=f"agent_{uuid.uuid4().hex[:8]}",
                name="health-search-agent",
                vector_store_id=vstore.id,
                instructions="""
You are a health resource advisor with access to dietary and recipe files.
You:
1. Always present disclaimers (you're not a doctor!)
2. Provide references to the files when possible
3. Focus on general nutrition or recipe tips.
4. Encourage professional consultation for more detailed advice.
"""
            )
            print(f"🎉 Created local health search agent, ID: {agent.id}")
            return agent
        else:
            # Server-side agent creation
            file_search_tool = FileSearchTool(vector_store_ids=[vstore.id])

            agent = project_client.agents.create_agent(
                model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-4o"),
                name="health-search-agent",
                instructions="""
You are a health resource advisor with access to dietary and recipe files.
You:
1. Always present disclaimers (you're not a doctor!)
2. Provide references to the files when possible
3. Focus on general nutrition or recipe tips.
4. Encourage professional consultation for more detailed advice.
""",
                tools=file_search_tool.definitions,
                tool_resources=file_search_tool.resources
            )
            print(f"🎉 Created health search agent (server-side), ID: {agent.id}")
            return agent
            
    except Exception as e:
        print(f"❌ Error creating health resource agent: {e}")
        return None

# Initialize our agent variable
health_agent = None

# Only create the agent if we successfully created a vector store earlier
if vector_store:
    health_agent = create_health_resource_agent(vector_store)

## 5. Searching Health Resources 🏋️👩‍🍳
We'll create a new conversation thread and ask queries like “Gluten-free recipe ideas?” or “Heart-healthy meal plan?” The agent will do file search on the vector store to find relevant info.

In [ ]:
@dataclass
class LocalSearchThread:
    """Local mock thread for search conversations"""
    id: str
    messages: List[dict]

def create_search_thread(agent):
    try:
        if isinstance(agent, LocalSearchAgent):
            # Create local thread
            thread = LocalSearchThread(
                id=f"thread_{uuid.uuid4().hex[:8]}",
                messages=[]
            )
            print(f"📝 Created local search thread, ID: {thread.id}")
            return thread
        else:
            # Server-side thread
            thread = project_client.agents.create_thread()
            print(f"📝 Created search thread (server-side), ID: {thread.id}")
            return thread
    except Exception as e:
        print(f"❌ Error creating search thread: {e}")
        return None

def search_local_vector_store(vector_store, query):
    """Simple local search through vector store content"""
    results = []
    query_lower = query.lower()
    
    for file_id, file_data in vector_store.file_contents.items():
        content = file_data['content'].lower()
        if query_lower in content or any(word in content for word in query_lower.split()):
            # Extract relevant snippet
            lines = file_data['content'].split('\n')
            relevant_lines = [l for l in lines if query_lower in l.lower() or any(w in l.lower() for w in query_lower.split())][:3]
            results.append({
                'file_id': file_id,
                'file_name': file_data['name'],
                'snippets': relevant_lines
            })
    
    return results

def ask_search_question(thread, agent, user_question, vector_store=None):
    try:
        if isinstance(agent, LocalSearchAgent) and isinstance(thread, LocalSearchThread):
            # Local search implementation
            print(f"🔎 Searching: '{user_question}'")
            
            # Store user message
            thread.messages.append({
                "role": "user",
                "content": user_question
            })
            
            # Search the local vector store
            search_results = search_local_vector_store(vector_store, user_question)
            
            # Generate response based on search results
            response = f"Based on the available resources:\n\n"
            
            if search_results:
                for result in search_results:
                    response += f"From {result['file_name']}:\n"
                    for snippet in result['snippets']:
                        response += f"- {snippet.strip()}\n"
                    response += "\n"
            else:
                response += "I couldn't find specific information matching your query in the available resources. "
            
            response += "\n⚠️ This is general information only. Please consult a healthcare professional for personalized advice."
            
            # Store assistant response
            thread.messages.append({
                "role": "assistant",
                "content": response
            })
            
            print(f"✅ Local search completed")
            return {"status": "completed", "type": "local"}
            
        else:
            # Server-side search
            message = project_client.agents.create_message(
                thread_id=thread.id,
                role="user",
                content=user_question
            )
            print(f"🔎 Searching: '{user_question}'")

            run = project_client.agents.create_and_process_run(
                thread_id=thread.id,
                agent_id=agent.id
            )
            print(f"🤖 Run finished with status: {run.status}")
            if run.last_error:
                print(f"Error details: {run.last_error}")
            return run
            
    except Exception as e:
        print(f"❌ Error searching question: {e}")
        return None

# Now let's test our search functionality!
if health_agent:
    search_thread = create_search_thread(health_agent)

    if search_thread:
        queries = [
            "Could you suggest a gluten-free lunch recipe?",
            "Show me some heart-healthy meal ideas.",
            "What guidelines do you have for someone with diabetes?"
        ]

        for q in queries:
            ask_search_question(search_thread, health_agent, q, vector_store)

## 6. View Results & Citations 📄
We'll read the conversation thread to see how the agent responded and see if it cited the correct files.

In [ ]:
def display_thread_messages(thread):
    try:
        if isinstance(thread, LocalSearchThread):
            # Display local thread messages
            print("\n🗣️ Conversation so far:")
            for m in thread.messages:
                print(f"{m['role'].upper()}: {m['content']}\n")
        else:
            # Display server-side thread messages
            messages = project_client.agents.list_messages(thread_id=thread.id)
            print("\n🗣️ Conversation so far:")
            for m in reversed(messages.data):
                if m.content:
                    last_content = m.content[-1]
                    if hasattr(last_content, "text"):
                        print(f"{m.role.upper()}: {last_content.text.value}\n")

            # Check for citations
            print("\n📎 Checking for citations...")
            if hasattr(messages, 'file_citation_annotations') and messages.file_citation_annotations:
                for c in messages.file_citation_annotations:
                    print(f"- Citation snippet: '{c.text}' from file ID: {c.file_citation['file_id']}")
            else:
                print("No citations found in this conversation.")

    except Exception as e:
        print(f"❌ Error displaying messages: {e}")

# Display the conversation history
if search_thread:
    display_thread_messages(search_thread)

## 7. Cleanup & Best Practices 🧹
We'll optionally remove the vector store, the uploaded files, and the agent. In a production environment, you might keep them around longer. Meanwhile, here are some tips:

1. **Resource Management**
   - Keep files grouped by category, regularly prune old or irrelevant files.
   - Clear out test agents or vector stores once you're done.

2. **Search Queries**
   - Provide precise or multi-part queries.
   - Consider synonyms or alternative keywords ("gluten-free" vs "celiac").
   
3. **Health Information**
   - Always disclaim that you are not a medical professional.
   - Encourage users to see doctors for specific diagnoses.

4. **Performance**
   - Keep an eye on vector store size.
   - Evaluate search accuracy with `azure-ai-evaluation`!


In [ ]:
def cleanup_all():
    try:
        # Handle vector store cleanup
        if 'vector_store' in globals() and vector_store:
            if isinstance(vector_store, LocalVectorStore):
                print("🗑️ Cleaned up local vector store (no server deletion needed).")
            else:
                try:
                    project_client.agents.delete_vector_store(vector_store.id)
                    print("🗑️ Deleted server-side vector store.")
                except Exception as e:
                    print(f"Note: Could not delete server-side vector store: {e}")

        # Remove uploaded files
        if 'file_ids' in globals() and file_ids:
            # Only try to delete server-side files if they exist
            if vector_store and not isinstance(vector_store, LocalVectorStore):
                for fid in file_ids:
                    try:
                        project_client.agents.delete_file(fid)
                    except:
                        pass
            print("🗑️ Handled file cleanup.")

        # Delete the AI agent
        if 'health_agent' in globals() and health_agent:
            if isinstance(health_agent, LocalSearchAgent):
                print("🗑️ Cleaned up local search agent (no server deletion needed).")
            else:
                try:
                    project_client.agents.delete_agent(health_agent.id)
                    print("🗑️ Deleted server-side search agent.")
                except Exception as e:
                    print(f"Note: Could not delete server-side agent: {e}")

        # Clean up local files
        if 'sample_files' in globals() and sample_files:
            for sf in sample_files:
                if os.path.exists(sf):
                    os.remove(sf)
            print("🗑️ Deleted local sample files.")

    except Exception as e:
        print(f"❌ Error during cleanup: {e}")

# Run our cleanup function
cleanup_all()

# Congratulations! 🎉
You've created a **Health Resource Search Agent** that:
1. Uses a **Vector Store** to store sample recipes & guidelines.
2. **Searches** them to answer queries.
3. **Provides disclaimers** reminding users to consult professionals.

Feel free to adapt this approach for your own corporate documents, product manuals, or custom health resources.

Happy Searching! 🎉

#### Let's proceed to [4-bing_grounding.ipynb](4-bing_grounding.ipynb)